# Benchmark Evaluator V4.0: Proposed Dual-Model Evaluator
### *Local GPU Evaluation for TelkomNusa-SLM-3B and TelkomNusa-LLM-7B*

---
### Karakteristik Utama Evaluator V4.0:
1. **Fokus Eksklusif Dual-Model Usulan**:
   - **TelkomNusa-SLM-3B-V2** (`Qwen2.5-3B-Instruct` + QLoRA BSS 4096)
   - **TelkomNusa-LLM-7B-V2** (`Qwen2.5-7B-Instruct` + QLoRA BSS 4096)
   - *(Bisa langsung diarahkan ke adapter hasil latih ulang balanced dataset jika sudah ada)*
2. **Bebas Kuota & Ketergantungan API Eksternal**:
   - Berjalan 100% lokal pada GPU Kaggle/Colab T4 16GB VRAM (Gratis).
3. **Pembersihan Total Residu Prompt Korea (Zero Contamination)**:
   - Membuang seluruh persona *counselor* Korea yang memicu halusinasi obrolan santai.
4. **Injeksi Entitas Nomor Pelanggan Aktif (*Stateful Injection*)**:
   - Mengekstrak otomatis nomor telepon pelanggan (`svc_num`) dari metadata persona pada dialog multi-turn via Regex dan menyuntikkannya ke *system instruction*.
5. **Manajemen Memori GPU Mandiri**:
   - Model 3B selesai dievaluasi -> VRAM dibersihkan tuntas (`gc.collect()` + `torch.cuda.empty_cache()`) -> Model 7B dimuat tanpa risiko OOM.


## Section 0: Setup Dependencies & Lingkungan Kerja GPU T4


In [ ]:
# 1. Bersihkan file residu lama
!rm -f /kaggle/working/*_v4_predictions.csv /kaggle/working/*Summary*.csv /kaggle/working/*.png /kaggle/working/*.zip

# 2. Pasang dependensi esensial (BitsAndBytes 4-bit, Accelerate, Evaluator NLG)
!pip install -q -U "bitsandbytes>=0.46.1" accelerate rouge-score tabulate

import os
import sys
import gc
import json
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Any, Tuple, Optional
from tqdm import tqdm

# Mencegah fragmentasi memori CUDA di GPU T4
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

try:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except Exception:
    pass

smoothie = SmoothingFunction().method4
rouge_evaluator = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# Hugging Face Token (Optional jika model adapter bersifat publik)
HF_TOKEN = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN_HERE")

BASE_OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./Hasil_Evaluasi_V4"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

print("=" * 80)
print(" SETUP EVALUATOR V4.0 SELESAI (DUAL-MODEL LOCAL GPU READY)")
print(f"• Folder Output : {BASE_OUTPUT_DIR}")
print(f"• Target Model  : TelkomNusa-SLM-3B-V2 & TelkomNusa-LLM-7B-V2")
print("=" * 80)


## Section 1: Pemuatan Dataset Held-Out Benchmark TelAgentBench Indonesia


In [ ]:
# 1. PEMUATAN DATASET BENCHMARK TELAGENTBENCH INDONESIA (REAL & BALANCED)
# STRUKTUR TEST SET EVALUASI BAB 4 SKRIPSI:
# 1. Single-Turn Tool Execution : 55 sampel butuh tool (telco_simple.json)
# 2. Single-Turn Abstention     : 55 sampel di luar lingkup (telco_live_irrelevance.json)
# Total Single-Turn          : 110 butir soal uji seimbang
# 3. Multi-Turn Interactive     : 46 dialog (telco_multi_turn_stateless_base.json)
# Total Uji Keseluruhan      : 156 sampel uji
import os
import json
import re

DATA_DIR = None
search_roots = ["/kaggle/input", "/content", "."]
for s_root in search_roots:
    if os.path.exists(s_root):
        for root, dirs, files in os.walk(s_root):
            if "TelAgent_Action" in dirs:
                DATA_DIR = root
                break
        if DATA_DIR:
            break

if DATA_DIR is None:
    raise FileNotFoundError("âŒ Folder benchmark 'TelAgent_Action' tidak ditemukan!")

ACTION_DIR = os.path.join(DATA_DIR, "TelAgent_Action") if "TelAgent_Action" not in DATA_DIR else DATA_DIR
print(f"[OK] [OK] Folder Benchmark Terdeteksi: {ACTION_DIR}")

SIMPLE_FILE    = os.path.join(ACTION_DIR, "telco_simple.json")
SIMPLE_ANS     = os.path.join(ACTION_DIR, "possible_answer", "telco_simple.json")
IRREL_FILE     = os.path.join(ACTION_DIR, "telco_live_irrelevance.json")
IRREL_ANS      = os.path.join(ACTION_DIR, "possible_answer", "telco_live_irrelevance.json")
MULTI_FILE     = os.path.join(ACTION_DIR, "telco_multi_turn_stateless_base.json")
MULTI_ANS      = os.path.join(ACTION_DIR, "possible_answer", "telco_multi_turn_stateless_base.json")
TOOLS_FILE     = os.path.join(ACTION_DIR, "250918_telcoFC_tools.json")

# 1. Pemuatan 23 Definisi Tool BSS
TELCO_TOOLS = []
with open(TOOLS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line_s = line.strip()
        if line_s:
            try:
                TELCO_TOOLS.append(json.loads(line_s))
            except Exception:
                pass
if not TELCO_TOOLS:
    with open(TOOLS_FILE, "r", encoding="utf-8") as f:
        TELCO_TOOLS = json.load(f)

FORMATTED_TOOLS = []
for t in TELCO_TOOLS:
    if "type" in t and t["type"] == "function":
        FORMATTED_TOOLS.append(t)
    elif "name" in t:
        FORMATTED_TOOLS.append({"type": "function", "function": t})

ALL_TOOL_NAMES = [t.get("function", t).get("name", "") for t in FORMATTED_TOOLS]

# 2. Pemuatan Soal Single-Turn: 55 Tool-Calling + 55 Abstention = 110 Soal
simple_instances = []
simple_answers = {}

# (a) 55 Sampel Tool Execution (Positive Samples)
with open(SIMPLE_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            item = json.loads(line)
            item["is_abstention"] = False
            simple_instances.append(item)

with open(SIMPLE_ANS, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            item = json.loads(line)
            calls = []
            for gt_item in item.get("ground_truth", []):
                if isinstance(gt_item, dict):
                    if "name" in gt_item:
                        calls.append(gt_item)
                    else:
                        for fn_name, fn_args in gt_item.items():
                            calls.append({"name": fn_name, "arguments": fn_args})
            simple_answers[item["id"]] = calls

# (b) 55 Sampel Abstention/Irrelevance (Negative Samples)
if os.path.exists(IRREL_FILE):
    with open(IRREL_FILE, "r", encoding="utf-8") as f:
        c = 0
        for line in f:
            if line.strip() and c < 55:
                item = json.loads(line)
                q_content = item.get("question", "")
                user_msg = ""
                if isinstance(q_content, list):
                    for turn in q_content:
                        if isinstance(turn, list):
                            for m in turn:
                                if m.get("role") == "user":
                                    user_msg = m.get("content", "")
                        elif isinstance(turn, dict) and turn.get("role") == "user":
                            user_msg = turn.get("content", "")
                simple_instances.append({
                    "id": item["id"],
                    "question": user_msg if user_msg else str(q_content),
                    "is_abstention": True
                })
                simple_answers[item["id"]] = []  # Ground Truth Kosong (Abstention)
                c += 1

# 3. Pemuatan Soal Multi-Turn (46 dialog)
multi_instances = []
with open(MULTI_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            multi_instances.append(json.loads(line))

multi_answers = {}
if os.path.exists(MULTI_ANS):
    with open(MULTI_ANS, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                all_turns = []
                for turn_list in item.get("ground_truth", []):
                    turn_calls = []
                    for call_str in turn_list:
                        s = str(call_str).strip()
                        m = re.match(r"^([A-Za-z0-9_]+__[\w\-]+)\((.*)\)$", s)
                        if m:
                            fname = m.group(1)
                            args_raw = m.group(2)
                            arg_dict = {}
                            for km, vm in re.findall(r"(\w+)\s*=\s*'([^']*)'", args_raw):
                                arg_dict[km] = vm
                            for km, vm in re.findall(r"(\w+)\s*=\s*(\d+)(?:\s*[,)]|\s*$)", args_raw):
                                if km not in arg_dict:
                                    arg_dict[km] = int(vm)
                            turn_calls.append({"name": fname, "arguments": arg_dict})
                    all_turns.append(turn_calls)
                multi_answers[item["id"]] = all_turns

print(f"\n [DATASET STATUS] Berhasil Dimuat Lengkap & Seimbang:")
print(f"â€¢ 23 Skema Tool API BSS        : {len(FORMATTED_TOOLS)} tools")
print(f"â€¢ Single-Turn (Tool Needed)    : {len([x for x in simple_instances if not x.get('is_abstention', False)])} soal")
print(f"â€¢ Single-Turn (Abstention Set) : {len([x for x in simple_instances if x.get('is_abstention', False)])} soal")
print(f"â€¢ Total Single-Turn Seimbang   : {len(simple_instances)} butir soal (50% Execution, 50% Abstention)")
print(f"â€¢ Multi-Turn Dialog            : {len(multi_instances)} dialog ({len(multi_answers)} ground-truth)")


## Section 2: Master Model Registry V4 (Khusus 2 Model Usulan: 3B vs 7B)


In [ ]:
# MODEL REGISTRY V4.0 (DUAL-MODEL USULAN: 3B & 7B BALANCED SFT VERSION)
# ==============================================================================\n# Model ini dilatih menggunakan dataset SFT TelkomNusa seimbang (3.129 data)
# dengan 600 sampel dialog multi-turn stateful via Unsloth Engine.

MODEL_REGISTRY = {
    # 1. Model Usulan Kompak (SLM 3B) - Balanced SFT
    "TelkomNusa-SLM-3B-Balanced": {
        "platform": "local_lora",
        "base_model": "Qwen/Qwen2.5-3B-Instruct",
        "adapter": "Rislantrs/telecom-troubleshooter-qwen2.5-3b-balanced-lora",
        "params": "3B",
        "category": "Proposed SLM (3B)",
        "regime": "Regime A (Zero-Shot SFT)",
        "mode": "Balanced SFT 2048",
        "is_few_shot": False,
        "active": True
    },

    # 2. Model Usulan Menengah (LLM 7B) - Balanced SFT
    "TelkomNusa-LLM-7B-Balanced": {
        "platform": "local_lora",
        "base_model": "Qwen/Qwen2.5-7B-Instruct",
        "adapter": "Rislantrs/telecom-troubleshooter-qwen2.5-7b-balanced-lora",
        "params": "7B",
        "category": "Proposed LLM (7B)",
        "regime": "Regime A (Zero-Shot SFT)",
        "mode": "Balanced SFT 2048",
        "is_few_shot": False,
        "active": True
    }
}

print(f" Evaluator V4.0: {len(MODEL_REGISTRY)} Model Usulan Balanced Terdaftar.")
for name, cfg in MODEL_REGISTRY.items():
    status = "AKTIF" if cfg.get("active", False) else "NONAKTIF"
    print(f"• [{cfg['params']:3s}] {name:28s} -> {status} (Adapter: {cfg['adapter']})")


## Section 3: Universal Parser, Normalizer & Evaluator Scoring Engine


In [ ]:
# 3. ENGINE EVALUATOR V3.4 (UNIVERSAL PARSER & PRECISE SCORER)

def extract_tool_calls(response_text: str) -> List[Dict[str, Any]]:
    calls = []
    if not response_text:
        return calls

    resp = response_text.replace('\\n', '\n')

    # 1. Format Tag XML Standar: <tool_call> ... </tool_call>
    xml_matches = re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", resp, re.DOTALL)
    for m in xml_matches:
        try:
            parsed = json.loads(m.strip())
            if isinstance(parsed, dict) and "name" in parsed:
                args = parsed.get("arguments", {})
                if isinstance(args, str):
                    try: args = json.loads(args)
                    except Exception: pass
                calls.append({"name": parsed["name"], "arguments": args if isinstance(args, dict) else {}})
        except Exception:
            pass
    if calls:
        return calls

    # 2. Format Markdown Code Block: ```json ... ```
    json_block = re.findall(r"```(?:json)?\s*([\[\{].*?[\]\}])\s*```", resp, re.DOTALL)
    for jb in json_block:
        try:
            parsed = json.loads(jb.strip())
            items = parsed if isinstance(parsed, list) else [parsed]
            for it in items:
                if isinstance(it, dict) and "name" in it:
                    args = it.get("arguments", {})
                    if isinstance(args, str):
                        try: args = json.loads(args)
                        except Exception: pass
                    calls.append({"name": it["name"], "arguments": args if isinstance(args, dict) else {}})
        except Exception:
            pass
    if calls:
        return calls

    # 3. Format Notasi Fungsi Klasik: FUNC__name(key='val', num=123)
    fn_matches = re.findall(r"([A-Za-z0-9_]+__[\w\-]+)\(([^)]*)\)", resp)
    for fname, args_str in fn_matches:
        arg_dict = {}
        for km, vm in re.findall(r"(\w+)\s*=\s*'([^']*)'", args_str):
            arg_dict[km] = vm
        for km, vm in re.findall(r"(\w+)\s*=\s*(\d+)(?:\s*[,)]|\s*$)", args_str):
            if km not in arg_dict:
                arg_dict[km] = int(vm)
        calls.append({"name": fname, "arguments": arg_dict})
    if calls:
        return calls

    # 4. Bare JSON
    bare_matches = re.findall(r'\{"name"\s*:\s*"([^"]+)"\s*,\s*"arguments"\s*:\s*(\{[^}]+\})\}', resp)
    for fname, args_raw in bare_matches:
        try:
            arg_dict = json.loads(args_raw)
            calls.append({"name": fname, "arguments": arg_dict if isinstance(arg_dict, dict) else {}})
        except Exception:
            calls.append({"name": fname, "arguments": {}})

    return calls

def normalize_tool_name(name: str) -> str:
    if not name:
        return ""
    return str(name).strip().lower().replace("-", "_")

def compare_tool_names(pred_name: str, gold_name: str) -> bool:
    return normalize_tool_name(pred_name) == normalize_tool_name(gold_name)

def compare_arguments(pred_args: Dict[str, Any], gold_args: Dict[str, Any]) -> bool:
    if not gold_args:
        return True
    if not pred_args:
        return False if gold_args else True

    for k, g_val in gold_args.items():
        if k not in pred_args:
            if isinstance(g_val, list) and (None in g_val or "" in g_val):
                continue
            return False

        p_val_str = str(pred_args[k]).strip().lower()
        if isinstance(g_val, list):
            allowed = [str(x).strip().lower() for x in g_val if x is not None]
            if allowed and p_val_str not in allowed:
                return False
        else:
            if p_val_str != str(g_val).strip().lower():
                return False
    return True

# SCORING SINGLE-TURN (DENGAN PEMISAHAN TOOL ACC & ABSTENTION ACC)
def score_single_turn_instance(pred_calls: List[Dict[str, Any]], gold_calls: List[Dict[str, Any]], is_abstention_sample: bool = False):
    # KASUS 1: SOAL ABSTENTION (Kueri di luar lingkup, gold_calls kosong)
    if not gold_calls or is_abstention_sample:
        if not pred_calls:
            # Sukses Abstain: model menolak dan tidak memanggil tool apa pun
            return 1.0, 1.0, 1.0, 1.0, 1.0, "Correct (Abstention)"
        else:
            # Halusinasi / Spurious Tool: model memanggil tool pada soal yang tidak relevan
            return 0.0, 0.0, 0.0, 1.0, 0.0, "Spurious Tool Call (Failed Abstention)"

    # KASUS 2: SOAL EKSEKUSI TOOL (Membutuhkan panggilan API BSS)
    if not pred_calls:
        return 0.0, 0.0, 0.0, 0.0, 0.0, "Format / No Tool Emitted"

    tool_match = compare_tool_names(pred_calls[0].get("name", ""), gold_calls[0].get("name", ""))
    if not tool_match:
        return 0.0, 0.0, 0.0, 1.0, 0.0, f"Wrong Tool (Pred: {pred_calls[0].get('name')} vs Gold: {gold_calls[0].get('name')})"

    arg_match = compare_arguments(pred_calls[0].get("arguments", {}), gold_calls[0].get("arguments", {}))
    if arg_match:
        return 1.0, 1.0, 1.0, 1.0, 1.0, "Correct"
    else:
        return 1.0, 0.0, 0.0, 1.0, 1.0, "Correct Tool, Wrong Arguments"

# SCORING MULTI-TURN TRUE PER-TURN (ANTI INDEX SHIFT)
def score_multi_turn_instance(pred_turns: Any, gold_turns: List[List[Dict[str, Any]]]):
    num_turns = len(gold_turns)
    if num_turns == 0:
        return 1.0, 1.0, 1.0, 1.0, "Correct (No Turns)"

    tool_scores, arg_scores, strict_scores = [], [], []
    total_calls_pred = 0

    is_nested = isinstance(pred_turns, list) and len(pred_turns) > 0 and isinstance(pred_turns[0], list)

    for t in range(num_turns):
        gold_turn_calls = gold_turns[t] if t < len(gold_turns) else []

        if is_nested:
            pred_turn_calls = pred_turns[t] if t < len(pred_turns) else []
        elif isinstance(pred_turns, list):
            pred_turn_calls = [pred_turns[t]] if t < len(pred_turns) else []
        else:
            pred_turn_calls = []

        total_calls_pred += len(pred_turn_calls)

        if not gold_turn_calls:
            if not pred_turn_calls:
                tool_scores.append(1.0); arg_scores.append(1.0); strict_scores.append(1.0)
            else:
                tool_scores.append(0.0); arg_scores.append(0.0); strict_scores.append(0.0)
            continue

        if not pred_turn_calls:
            tool_scores.append(0.0); arg_scores.append(0.0); strict_scores.append(0.0)
            continue

        t_matches = 0
        s_matches = 0
        for g_call in gold_turn_calls:
            matched_tool = False
            matched_strict = False
            for p_call in pred_turn_calls:
                if compare_tool_names(p_call.get("name", ""), g_call.get("name", "")):
                    matched_tool = True
                    if compare_arguments(p_call.get("arguments", {}), g_call.get("arguments", {})):
                        matched_strict = True
                        break
            if matched_tool:
                t_matches += 1
            if matched_strict:
                s_matches += 1

        turn_t_acc = t_matches / len(gold_turn_calls)
        turn_s_ast = s_matches / len(gold_turn_calls)
        turn_a_m = turn_s_ast if t_matches > 0 else 0.0

        tool_scores.append(turn_t_acc)
        arg_scores.append(turn_a_m)
        strict_scores.append(turn_s_ast)

    avg_tool = sum(tool_scores) / num_turns
    avg_arg = sum(arg_scores) / num_turns
    avg_strict = sum(strict_scores) / num_turns
    fmt_valid = 1.0 if total_calls_pred > 0 else 0.0
    return avg_tool, avg_arg, avg_strict, fmt_valid, f"Per-Turn Evaluated ({num_turns} Turns)"

print("[OK] Engine Evaluator V3.4 Siap.")

# METRIK GENERASI BAHASA ALAMI (BLEU-4 & ROUGE-L)
def compute_nlg_metrics(pred_text: str, gold_calls: list) -> Tuple[float, float]:
    if not pred_text or not gold_calls:
        return 0.0, 0.0

    ref_parts = []
    for gc in gold_calls:
        if isinstance(gc, dict):
            ref_parts.append(json.dumps(gc, sort_keys=True, ensure_ascii=False))
        elif isinstance(gc, list):
            for g in gc:
                if isinstance(g, dict):
                    ref_parts.append(json.dumps(g, sort_keys=True, ensure_ascii=False))
    ref_text = " ".join(ref_parts)

    if not ref_text.strip():
        return 0.0, 0.0

    try:
        ref_tokens = ref_text.lower().split()
        pred_tokens = pred_text.lower().split()
        bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothie)
    except Exception:
        bleu = 0.0

    try:
        rouge_result = rouge_evaluator.score(ref_text, pred_text)
        rouge_l = rouge_result['rougeL'].fmeasure
    except Exception:
        rouge_l = 0.0

    return round(float(bleu) * 100.0, 2), round(float(rouge_l) * 100.0, 2)


## Section 4: Inference Engine V4.0 (Pure Clean Prompt + Regex SvcNum Injection)


In [ ]:
# 4. INFERENCE ENGINE V4.0 (CLEAN LOCAL PROMPTING + REGEX ACTIVE SVC_NUM)
# KEUNGGULAN V4.0:
# 1. LOCAL_SYSTEM_PROMPT murni (identik kondisi latih SFT).
# 2. Multi-turn: Mengambil nomor HP pelanggan ('svc_num') secara otomatis via regex
# dari persona soal dan menyuntikkannya sebagai instruksi aktif sistem.
# 3. Membuang 100% kata-kata konselor telepon ramah Korea yang merusak fungsi agentic.
# 4. Pembersihan memori GPU antar-model otomatis untuk stabilitas T4 VRAM.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

LOCAL_SYSTEM_PROMPT = "Anda adalah asisten AI resmi Customer Care TelkomNusa. Panggil tool yang sesuai jika diperlukan."

CURRENT_MODEL = None
CURRENT_TOKENIZER = None
CURRENT_MODEL_KEY = None

def unload_current_local_model():
    """Bersihkan VRAM secara menyeluruh sebelum memuat model berikutnya."""
    global CURRENT_MODEL, CURRENT_TOKENIZER, CURRENT_MODEL_KEY
    if CURRENT_MODEL is not None:
        del CURRENT_MODEL
        CURRENT_MODEL = None
    if CURRENT_TOKENIZER is not None:
        del CURRENT_TOKENIZER
        CURRENT_TOKENIZER = None
    CURRENT_MODEL_KEY = None
    gc.collect()
    torch.cuda.empty_cache()
    print(" [GPU VRAM] Memori GPU dibersihkan tuntas.")

def load_local_lora_model(base_model_name: str, adapter_name: str) -> Tuple[Any, Any]:
    """Pemuatan model 4-bit + adapter LoRA."""
    global CURRENT_MODEL, CURRENT_TOKENIZER, CURRENT_MODEL_KEY
    key = f"{base_model_name}::{adapter_name}"
    if CURRENT_MODEL_KEY == key and CURRENT_MODEL is not None:
        return CURRENT_MODEL, CURRENT_TOKENIZER

    unload_current_local_model()
    print(f"\n⏳ [GPU VRAM] Memuat Base Model 4-bit: {base_model_name}...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

    tokenizer = AutoTokenizer.from_pretrained(base_model_name, token=HF_TOKEN, trust_remote_code=True)
    base_m = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        token=HF_TOKEN,
        trust_remote_code=True
    )

    print(f"⏳ [GPU VRAM] Menggabungkan LoRA Adapter: {adapter_name}...")
    model = PeftModel.from_pretrained(base_m, adapter_name, token=HF_TOKEN)
    model.eval()

    CURRENT_MODEL = model
    CURRENT_TOKENIZER = tokenizer
    CURRENT_MODEL_KEY = key
    print(f" [BERHASIL] Model {base_model_name} + Adapter {adapter_name} Siap di GPU VRAM!")
    return CURRENT_MODEL, CURRENT_TOKENIZER

def run_local_lora_inference(messages: List[Dict[str, str]], base_model_name: str, adapter_name: str) -> Tuple[str, float]:
    """Inferensi lokal LoRA: bersih, tanpa prompt bloat, fast EOS stopping."""
    model, tokenizer = load_local_lora_model(base_model_name, adapter_name)
    try:
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tools=FORMATTED_TOOLS,
            tokenize=False,
            add_generation_prompt=True
        )
    except Exception:
        prompt_text = ""
        for m in messages:
            prompt_text += f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n"
        prompt_text += "<|im_start|>assistant\n"

    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

    eos_ids = [tokenizer.eos_token_id]
    for extra_tok in ["<|im_end|>", "<|endoftext|>"]:
        try:
            tid = tokenizer.convert_tokens_to_ids(extra_tok)
            if isinstance(tid, int) and tid > 0:
                eos_ids.append(tid)
        except Exception:
            pass
    eos_ids = list(set(eos_ids))

    t0 = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.01,
            top_p=0.95,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=eos_ids
        )
    lat = time.time() - t0
    gen_tokens = output_ids[0][inputs["input_ids"].shape[1]:]

    resp_text = tokenizer.decode(gen_tokens, skip_special_tokens=False).strip()
    for stop_tok in ["<|im_end|>", "<|endoftext|>", "</s>"]:
        resp_text = resp_text.replace(stop_tok, "").strip()

    return resp_text, lat

def evaluate_single_model(model_name: str, config: Dict[str, Any]) -> Dict[str, Any]:
    print("\n" + "=" * 80)
    print(f" [EVALUASI V4.0] {model_name} [{config.get('params', 'N/A')}]")
    print(f"• Base: {config.get('base_model')} | Adapter: {config.get('adapter')}")
    print("=" * 80)
    output_pred_csv = os.path.join(BASE_OUTPUT_DIR, f"{model_name}_v4_predictions.csv")

    st_records, mt_records = [], []
    all_bleu, all_rouge = [], []
    st_abstention_scores = []
    st_tool_scores = []
    st_strict_scores = []
    st_arg_scores = []

    # 1. EKSEKUSI SINGLE-TURN (110 Soal: 55 Tool Execution + 55 Abstention)
    for inst in tqdm(simple_instances, desc=f"{model_name} [Single-Turn]"):
        q_id = inst["id"]
        q_content = inst["question"]
        gold = simple_answers.get(q_id, [])
        is_abst = inst.get("is_abstention", False)

        messages = [{"role": "system", "content": LOCAL_SYSTEM_PROMPT}]
        if isinstance(q_content, list):
            for m_item in q_content:
                if isinstance(m_item, list):
                    messages.extend(m_item)
                else:
                    messages.append(m_item)
        else:
            messages.append({"role": "user", "content": str(q_content)})

        resp, lat = run_local_lora_inference(messages, config["base_model"], config["adapter"])

        p_calls = extract_tool_calls(resp)
        t_acc, a_m, s_ast, f_val, abst_acc, cat = score_single_turn_instance(p_calls, gold, is_abstention_sample=is_abst)

        if is_abst:
            st_abstention_scores.append(abst_acc)
        else:
            st_tool_scores.append(t_acc)
            st_arg_scores.append(a_m)
            st_strict_scores.append(s_ast)

        bleu_val, rouge_val = compute_nlg_metrics(resp, gold)
        all_bleu.append(bleu_val)
        all_rouge.append(rouge_val)

        st_records.append({
            "id": q_id, "type": "Single-Turn", "is_abstention": is_abst, "raw_response": resp,
            "tool_accuracy": t_acc, "arg_match": a_m, "strict_ast": s_ast,
            "format_valid": f_val, "abstention_acc": abst_acc, "category": cat,
            "latency_s": lat, "bleu4": bleu_val, "rouge_l": rouge_val
        })

    # 2. EKSEKUSI MULTI-TURN (47 Percakapan Interaktif - CLEAN PROMPT & REGEX SVC)
    for inst in tqdm(multi_instances, desc=f"{model_name} [Multi-Turn]"):
        q_id = inst["id"]
        q_content = inst["question"]
        gold_t = multi_answers.get(q_id, [])

        # 1. Ekstrak data nomor pelanggan (svc_num) dari persona JSON bawaan soal
        raw_sys = ""
        if isinstance(q_content, list) and len(q_content) > 0 and isinstance(q_content[0], list):
            for tm in q_content[0]:
                if tm.get("role") == "system":
                    raw_sys = tm.get("content", "")
                    break

        # Ekstrak svc_num secara otomatis menggunakan Regex
        svc_match = re.search(r'"svc_num"\s*:\s*"([^"]+)"', raw_sys)
        active_svc_num = svc_match.group(1) if svc_match else ""

        # 2. Rancang System Prompt Bersih (Bebas instruksi konselor ramah Korea!)
        clean_sys_prompt = "Anda adalah asisten AI resmi Customer Care TelkomNusa. Panggil tool yang sesuai jika diperlukan."
        if active_svc_num:
            clean_sys_prompt += f"\n[INFORMASI PELANGGAN AKTIF]: Nomor layanan pelanggan yang sedang dilayani adalah {active_svc_num}. Gunakan nomor ini sebagai parameter svcMgmtNum/serviceManagementNumber pada pemanggilan API terkait."
        history = [{"role": "system", "content": clean_sys_prompt}]

        pred_turns = []
        turn_latencies = []
        turn_responses = []

        if isinstance(q_content, list) and len(q_content) > 0 and isinstance(q_content[0], list):
            for turn_messages in q_content:
                for tm in turn_messages:
                    if tm.get("role") != "system":
                        history.append(tm)
                t_resp, t_lat = run_local_lora_inference(history, config["base_model"], config["adapter"])
                turn_latencies.append(t_lat)
                turn_responses.append(t_resp)
                history.append({"role": "assistant", "content": t_resp})

                t_calls = extract_tool_calls(t_resp)
                pred_turns.append(t_calls)
        else:
            msgs_flat = [{"role": "system", "content": history[0]["content"]}]
            msgs_flat.append({"role": "user", "content": str(q_content)})
            resp, lat = run_local_lora_inference(msgs_flat, config["base_model"], config["adapter"])
            t_calls = extract_tool_calls(resp)
            pred_turns = [t_calls]
            turn_latencies = [lat]
            turn_responses = [resp]

        t_acc, a_m, s_ast, f_val, cat = score_multi_turn_instance(pred_turns, gold_t)

        mt_resp_combined = " ".join(turn_responses)
        mt_gold_flat = []
        for gt in gold_t:
            if isinstance(gt, list):
                mt_gold_flat.extend(gt)
            elif isinstance(gt, dict):
                mt_gold_flat.append(gt)

        bleu_val, rouge_val = compute_nlg_metrics(mt_resp_combined, mt_gold_flat)
        all_bleu.append(bleu_val)
        all_rouge.append(rouge_val)

        mt_records.append({
            "id": q_id, "type": "Multi-Turn", "raw_response": json.dumps(pred_turns),
            "tool_accuracy": t_acc, "arg_match": a_m, "strict_ast": s_ast,
            "format_valid": f_val, "category": cat,
            "latency_s": np.mean(turn_latencies) if turn_latencies else 0.1,
            "bleu4": bleu_val, "rouge_l": rouge_val
        })

    df_all = pd.DataFrame(st_records + mt_records)
    df_all.to_csv(output_pred_csv, index=False)
    print(ff" File Prediksi Lengkap Disimpan: {output_pred_csv}")

    # Bersihkan memori GPU setelah selesai satu model
    unload_current_local_model()

    # Kalkulasi Ringkasan Evaluation Metrics
    st_df = df_all[df_all["type"] == "Single-Turn"]
    mt_df = df_all[df_all["type"] == "Multi-Turn"]

    st_tool_only = st_df[st_df["is_abstention"] == False]
    st_abst_only = st_df[st_df["is_abstention"] == True]

    res_row = {
        "Model Name": model_name,
        "Params": config.get("params", "N/A"),
        "Category": config.get("category", "Proposed"),
        "Regime": config.get("regime", "Regime A"),
        "Overall Strict AST (%)": round(df_all["strict_ast"].mean() * 100, 2),
        "ST Tool Acc (%)": round(st_tool_only["tool_accuracy"].mean() * 100, 2) if len(st_tool_only) > 0 else 0.0,
        "ST Arg Match (%)": round(st_tool_only["arg_match"].mean() * 100, 2) if len(st_tool_only) > 0 else 0.0,
        "ST Strict AST (%)": round(st_tool_only["strict_ast"].mean() * 100, 2) if len(st_tool_only) > 0 else 0.0,
        "Abstention Acc (%)": round(st_abst_only["abstention_acc"].mean() * 100, 2) if len(st_abst_only) > 0 else 0.0,
        "MT Tool Acc (%)": round(mt_df["tool_accuracy"].mean() * 100, 2) if len(mt_df) > 0 else 0.0,
        "MT Arg Match (%)": round(mt_df["arg_match"].mean() * 100, 2) if len(mt_df) > 0 else 0.0,
        "MT Strict AST (%)": round(mt_df["strict_ast"].mean() * 100, 2) if len(mt_df) > 0 else 0.0,
        "Format Valid (%)": round(df_all["format_valid"].mean() * 100, 2),
        "BLEU-4": round(np.mean(all_bleu), 4),
        "ROUGE-L": round(np.mean(all_rouge), 4),
        "Latency (s)": round(df_all["latency_s"].mean(), 2)
    }
    return res_row


## Section 5: Tabel Komparasi 10 Pilar Dual-Model (Format Bab 4 Skripsi / IEEE)


In [ ]:
# 5. EKSEKUSI REKAPITULASI Evaluation Metrics DUAL-MODEL
all_results = []

for model_name, config in MODEL_REGISTRY.items():
    if config.get("active", False):
        try:
            res = evaluate_single_model(model_name, config)
            all_results.append(res)
        except Exception as e:
            print(f" Gagal mengevaluasi {model_name}: {e}")

df_comparison = pd.DataFrame(all_results)

if "Overall Strict AST (%)" in df_comparison.columns:
    df_comparison = df_comparison.sort_values(by="Overall Strict AST (%)", ascending=False).reset_index(drop=True)

print("\n" + "=" * 130)
print(" TABEL KOMPARASI Evaluation Metrics V4.0 (DUAL-MODEL TELKOMNUSA USULAN):")
print("=" * 130)
print(df_comparison.to_string(index=False))
print("=" * 130)

MASTER_SUMMARY_CSV = os.path.join(BASE_OUTPUT_DIR, "V4_TelkomNusa_DualModel_Benchmark_Summary.csv")
df_comparison.to_csv(MASTER_SUMMARY_CSV, index=False)
print(f"\n Master Summary CSV Disimpan di: {MASTER_SUMMARY_CSV}")


## Section 6: Visualisasi Grafik Komparasi 3B vs 7B Beresolusi Tinggi (300 DPI)


In [ ]:
# 6. VISUALISASI GRAFIK KOMPARASI 3B VS 7B (300 DPI)
print("[*] Merender Grafik Ilmiah Komparasi 3B vs 7B...")

plt.rcParams.update({"font.size": 10, "axes.labelsize": 11, "axes.titlesize": 12, "figure.titlesize": 14})

if len(df_comparison) > 0:
    # GRAFIK 1: Side-by-Side Bar Chart Pilar Kunci
    key_metrics = ["Overall Strict AST (%)", "ST Tool Acc (%)", "Abstention Acc (%)", "MT Tool Acc (%)", "MT Strict AST (%)", "Format Valid (%)"]
    fig, ax = plt.subplots(figsize=(12, 6), dpi=300)
    x = np.arange(len(key_metrics))
    width = 0.35

    colors = ["#1f77b4", "#2ca02c"]
    for i, row in df_comparison.iterrows():
        vals = [row[m] for m in key_metrics]
        ax.bar(x + (i - 0.5) * width, vals, width, label=f"{row['Model Name']} ({row['Params']})", color=colors[i % len(colors)])

    ax.set_ylabel("Skor Akurasi (%)")
    ax.set_title("Perbandingan 6 Pilar Kunci: TelkomNusa-SLM-3B vs TelkomNusa-LLM-7B")
    ax.set_xticks(x)
    ax.set_xticklabels(key_metrics, rotation=20, ha="right")
    ax.set_ylim(0, 105)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.legend(loc="upper left")
    plt.tight_layout()
    g1_path = os.path.join(BASE_OUTPUT_DIR, "Grafik1_DualModel_Pilar_Comparison.png")
    plt.savefig(g1_path, dpi=300)
    plt.close()
    print(f"• Tersimpan: {g1_path}")

    # GRAFIK 2: Single-Turn vs Multi-Turn Accuracy
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    st_vs_mt = ["ST Strict AST (%)", "MT Strict AST (%)"]
    x2 = np.arange(len(st_vs_mt))
    for i, row in df_comparison.iterrows():
        vals = [row[m] for m in st_vs_mt]
        ax.bar(x2 + (i - 0.5) * width, vals, width, label=f"{row['Model Name']}", color=colors[i % len(colors)])
    ax.set_ylabel("Strict AST Match (%)")
    ax.set_title("Ketahanan Generalisasi: Single-Turn vs Multi-Turn Interactive")
    ax.set_xticks(x2)
    ax.set_xticklabels(["Single-Turn Strict AST", "Multi-Turn Strict AST"])
    ax.set_ylim(0, 105)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.legend()
    plt.tight_layout()
    g2_path = os.path.join(BASE_OUTPUT_DIR, "Grafik2_Single_vs_MultiTurn_StrictAST.png")
    plt.savefig(g2_path, dpi=300)
    plt.close()
    print(f"• Tersimpan: {g2_path}")

print(" Grafik Ilmiah selesai dibuat.")


## Section 7: Ringkasan Eksekutif & Auto-Zip Export


In [ ]:
# 7. RINGKASAN EKSEKUTIF & AUTO-ZIP EXPORT
import zipfile

print("\n" + "=" * 90)
print(" RINGKASAN PERFORMA EVALUASI V4.0 (DUAL-MODEL USULAN):")
print("=" * 90)
if len(df_comparison) > 0:
    for idx, row in df_comparison.iterrows():
        print(f"{idx+1}. {row['Model Name']} [{row['Params']}]:")
        print(f"• Overall Strict AST : {row['Overall Strict AST (%)']}%")
        print(f"• Single-Turn Tool   : {row['ST Tool Acc (%)']}%")
        print(f"• Multi-Turn Tool    : {row['MT Tool Acc (%)']}%")
        print(f"• Latensi Rata-Rata  : {row['Latency (s)']} s/query")
print("=" * 90)

ZIP_EXPORT_PATH = os.path.join(BASE_OUTPUT_DIR, "V4_TelkomNusa_DualModel_Benchmark_Results.zip")
with zipfile.ZipFile(ZIP_EXPORT_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(BASE_OUTPUT_DIR):
        for f in files:
            if f.endswith(('.csv', '.png', '.json')) and not f.startswith('V4_TelkomNusa_DualModel_Benchmark_Results'):
                f_full = os.path.join(root, f)
                zipf.write(f_full, arcname=f)

print(f"\n Seluruh Hasil Evaluasi V4.0 Berhasil Dibundel:")
print(f"{ZIP_EXPORT_PATH} ({round(os.path.getsize(ZIP_EXPORT_PATH) / 1024, 1)} KB)")
print(" EVALUASI DUAL-MODEL TELKOMNUSA V4.0 selesai.")
